In [2]:
from geneformer.tokenizer import DEVICE
device = DEVICE


# In silico perturbation experiment

In [3]:
from geneformer import InSilicoPerturber
from geneformer import InSilicoPerturberStats

from geneformer.tokenizer import TOKEN_DICTIONARY_FILE
from geneformer.in_silico_perturber_stats import GENE_NAME_ID_DICTIONARY_FILE
from geneformer.in_silico_perturber import ISP_device

import sys

print(f"usse gpu num: {ISP_device}")
print(f"gene to ens dict: {GENE_NAME_ID_DICTIONARY_FILE}")
print(f"token directory : {TOKEN_DICTIONARY_FILE}")


usse gpu num: mps
gene to ens dict: /Users/petadimensionlab/workspace/Mouse-Geneformer-MPS/data/Mouse-Genecorpus-20M/MLM-re_token_dictionary_v1_GeneSymbol_to_EnsemblID.pkl
token directory : /Users/petadimensionlab/workspace/Mouse-Geneformer-MPS/data/Mouse-Genecorpus-20M/MLM-re_token_dictionary_v1.pkl


In [5]:
from datasets import load_from_disk
import numpy as np


# load disease dataset (xxx.dataset)
dataset_name = "./data/Mouse-Genecorpus-20M/eval_dataset/in_silico_perturbation/kidney_isp_mouse_tokenize_dataset_v-n1.dataset"


data = load_from_disk(dataset_name)
print(data)
print("total cells: {}".format(len(data["length"])))

disease_types = np.unique(data["disease"])
print(disease_types)
print(disease_types.shape[0])




Dataset({
    features: ['input_ids', 'cell_types', 'organ_major', 'disease', 'length'],
    num_rows: 72638
})
total cells: 72638
['autosomal dominant polycystic kidney disease' 'diabetic kidney disease'
 'normal']
3


In [6]:
# in silico perturbation in deletion mode to determine genes whose 
# deletion in the dilated cardiomyopathy (dcm) state significantly shifts
# the embedding towards non-failing (nf) state


# select_perturb_type: "delete","overexpress","inhibit","activate"

# "delete": delete gene from rank value encoding
# "overexpress": move gene to front of rank value encoding
# "inhibit": move gene to lower quartile of rank value encoding
# "activate": move gene to higher quartile of rank value encoding


select_perturb_type="delete" # select perturb type in delete, overexpress, inhibit or activate, you wanna it.

organ_data = "kidney"
start_state = 'normal'
end_state = 'diabetic kidney disease'
alt_state = [] # If alt state is nothing, leave list empty.

# select model type in Pretrained, GeneClassifier or CellClassifier, you wanna it.
use_model_type = "Pretrained"
num_classes = 0

# If you wanna perturb genes in your dataset, append its Ensembl ID in the list.
genes_to_perturb_list = [] # 

isp = InSilicoPerturber(perturb_type=select_perturb_type,
                        perturb_rank_shift=None,
                        genes_to_perturb="all" if len(genes_to_perturb_list) == 0 else genes_to_perturb_list,
                        combos=0,
                        anchor_gene=None,
                        model_type=use_model_type,
                        num_classes=disease_types.shape[0],
                        emb_mode="cell",
                        cell_emb_style="mean_pool",
                        filter_data=None,
                        cell_states_to_model={'state_key': 'disease', 
                                              'start_state':start_state, 
                                              'goal_state':end_state, 
                                              'alt_states': alt_state}, 
                        max_ncells=2000,
                        emb_layer=0,
                        forward_batch_size=50,
                        nproc=8)



In [5]:
# outputs intermediate files from in silico perturbation

start_state = start_state.replace(" ","-")
end_state = end_state.replace(" ","-")

import os   
DIR_NAME = "/Users/petadimensionlab/workspace/Mouse-Geneformer-MPS/results"
if not os.path.exists(DIR_NAME):
    os.mkdir(DIR_NAME)

isp.perturb_data("/Users/petadimensionlab/workspace/Mouse-Geneformer-MPS/mouse-Geneformer-L12-E20/", # If you choice "CellClassifier", you choice fine tuning model. If you choice "Pretrained", you choice pretrained model.
                 dataset_name, # input data
                 "/Users/petadimensionlab/workspace/Mouse-Geneformer-MPS/results/",
                 "output_in-silico_SE{}_OR{}_ST{}_EN{}".format(select_perturb_type, organ_data, start_state, end_state)) # output prefix


Filter (num_proc=8):   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/72638 [00:00<?, ? examples/s]

/Users/petadimensionlab/workspace/Mouse-Geneformer-MPS/.venv/lib/python3.12/site-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in SortOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)


  0%|          | 0/2000 [00:00<?, ?it/s]

...

Map (num_proc=8):   0%|          | 0/200 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/200 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/200 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/200 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/200 [00:00<?, ? examples/s]

In [7]:
ispstats = InSilicoPerturberStats(mode="goal_state_shift",
                                  genes_perturbed="all" if len(genes_to_perturb_list) == 0 else genes_to_perturb_list,
                                  combos=0,
                                  anchor_gene=None,
                                  cell_states_to_model={"state_key": "disease",
                                                        "start_state": start_state,
                                                        "goal_state": end_state,
                                                        "alt_states": alt_state},
                                 )


In [ ]:
# extracts data from intermediate files and processes stats to output in final .csv

start_state = start_state.replace(" ","-")
end_state = end_state.replace(" ","-")

import os
DIR_NAME = "/Users/petadimensionlab/workspace/Mouse-Geneformer-MPS/results/ispstats/"
if not os.path.exists(DIR_NAME):
    os.mkdir(DIR_NAME)

ispstats.get_stats("/Users/petadimensionlab/workspace/Mouse-Geneformer-MPS/results", # path to input data
                   None,
                   "/Users/petadimensionlab/workspace/Mouse-Geneformer-MPS/results/ispstats/", 
                   "output_in-silico_SE{}_OR{}_ST{}_EN{}".format(select_perturb_type, organ_data, start_state, end_state)) # output prefix 


  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/15912 [00:00<?, ?it/s]

  0%|          | 0/15912 [00:00<?, ?it/s]